In [1]:
import random
import pandas as pd
import numpy as np
from deap import base, creator, tools, algorithms
import joblib
import itertools

In [2]:
# 假设您已经有了预训练的模型文件
model_optimalTR = joblib.load("model_optimalsameTR.pkl")
model_optimalTs = joblib.load("model_optimalmixTS.pkl")

creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()

# 1

In [3]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(0,1000),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 90.0
alpha = 0.95
schedule = annealing_schedule(initial_temperature, alpha, n_gen)

In [4]:
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.18638567902965186, 0.15504756150063898, 0.028619935157343346, 0.021116377139941432, 0.005336128806498668, 33.82143840057355, 25.6, 5.048054920362675, 2.2250552961612344, 2.9030533489412718, 0.00258594403448655, 1.7074203529633885, 0.8627620942696715, 0.28, 0.0004300807214499204, 0.5431368032166185, 0.006757440384253305, 2.4767708076237303, 4.862426453381791, 14.03453878384332, 0.0023125682200926647, 6.4235533234007915, 1.40414573752043, 0.9486660290632405, 1.6646677204426346, 0.6139732546502331, 6.561636836518416, 1.0559236830290142, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 82.55459035599077, 8.20096644387093, 0.07838578522037336, 0.4317600883161104, 0.9295197239885974, 0.5818728207343822, 0.917941008628143, 0.009867021931749098, 0.575862663033905, 0.6617944251278588, 0.6092616783817664, 0.47894439253800053, 0.2050206627728592, 0.5600580337265518, 0.5929579755151315, 0.3148734078690963, 1, 0.22337563217714662, 0.050985410724532804, 0, 0.8956

In [6]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A1.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A1.xlsx


# 探索

In [9]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.09401455571154073, 0.7276862030907231, 1.71, 0.01928411377581912, 0.01524584232111664, 51.362321559056, 10.04312750128565, 0.8372987496717389, 2.48, 1.0302137777679152, 0.0057824422381907575, 3.069753308771984, 1.8289943190562832, 0.025774993879502595, 0.006943794155999883, 0, 0.00902311077148283, 4.325036650284522, 14.380864042245816, 1.7002286344261077, 0.02048446376566222, 6.768085167675148, 1.033809270035964, 0.01419605727081373, 1.6039193898792543, 0.7484921279796324, 7.803164897112129, 1.0091229529450638, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 27.354212917376366, 8.284219675071931, 0.22201605524154724, 0.6645119902341288, 0.6761600249090228, 0.5334177814906678, 0.6815279664033993, 0.49329122104065015, 0.2167800335724356, 0.9340731967033973, 0.04396497356674988, 0.07173026728103102, 0.8883813751189324, 0.9632362990824354, 0.025141044200038428, 0.9786869339199898, 0.49268165383507256, 0.5532693985006655, 0.7143939258943527, 0.323736337

In [10]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索1.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索1.xlsx


## 500-600

In [11]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(500,600),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.4219780056187536, 0.8053142146641353, 0.005134622241797234, 0.022804411845096365, 0.0032730750697413367, 24.995005152188316, 16.9229091029154, 0.6363150734709335, 2.1993868768324862, 1.4856857959488985, 0.0056898591847382, 0.02940850653526086, 1.0448261514095096, 0.027586695480597143, 0.275499955121822, 3.5055237127684976e-05, 0.005097354453604106, 0.6551632338409887, 14.356129161580263, 6.099434948850538, 0.0036627546309487896, 5.843963672120677, 1.8319188341691868, 0.9319560929514668, 1.8143345084843687, 0.8746461727595852, 9.52599203884628, 1.1829981386110708, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 70.97755284078201, 5.413889505791155, 0.051904603435180025, 0.19664653155605463, 0.9366248827375692, 0.46355339309928734, 0.24896963842970823, 0.371399683699683, 0.9920900838872959, 0.37143225013903336, 0.9639290264281223, 0.12860723516396144, 0.31698428122767136, 0.9452179579838368, 0.2435727582351405, 0.7917744064837181, 0.5445437432763867,

In [12]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索500-600.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索500-600.xlsx


## 600-700

In [13]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(600,700),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2525893464309884, 0.13512877929653297, 0.08996980490180222, 0.019013955834431853, 0.0009109438902816387, 2.6032643327023433, 24.863205989323795, 5.068482873346452, 2.3077483924352507, 2.7257926702676536, 0.0024517365721123396, 0.2608854668641132, 2.193921233705372, 0.28, 0.12784283995158058, 0.88, 0.0003329785063215648, 1.5714482112992305, 11.297172781423864, 8.356276826536032, 0.002012908108507951, 8.157445086834151, 1.5604633622036397, 0.8183243426327904, 1.8712900179748624, 0.9910473445447443, 8.906672480195146, 2.9537770009273547, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 88.81909596034289, 6.326896413216794, 0.7868884248916128, 0.0037842136804690438, 0.9744945882484467, 0.750045921135268, 0.7613090079613475, 0.3103286684135908, 0.6702059099984002, 0.8782321249729738, 0.4038971815548415, 0.832061536511543, 0.07438476714369902, 0.5064662049943915, 0.018769823108394205, 0.9607945007778547, 0.7134861458898438, 0.16844794086222636, 0.04074676

In [14]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索600-700.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索600-700.xlsx


## 700-800

In [15]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(700,800),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.42047420184021783, 0.6141959063955352, 1.71, 0.002000648366272323, 0.019174388715841977, 41.528342414012265, 5.675992074760514, 5.039732154352036, 0.7539734302498762, 2.607519509707702, 0.002818259636190408, 3.018415726766243, 2.0070236565201687, 0.02625730286027903, 0.2757991160570688, 0.5548524389603788, 0.006226971465416495, 2.531550877528343, 1.3116280568788727, 28.88, 0.023411976750170604, 5.049614883769862, 1.4420277079132249, 0.4900135464033945, 1.098419247656366, 0.7287609268784155, 7.575933138423015, 2.9814317841629885, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 75.57832000695858, 1.442634712819158, 0.8927629788364655, 0.41008507916265013, 0.759784586823639, 0.9986739871956916, 0.16124780498175043, 0.4246157457234869, 0.09308702234130573, 0.6248984957680065, 0.7440620097237566, 0.02932315442653895, 0.47454103886646093, 0.9637887742770543, 0.26272363724868186, 0.186859192410331, 0.06477136750282714, 0.19561434458179902, 0.9908054702760

In [16]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索700-800.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索700-800.xlsx


## 800-900

In [17]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(800,900),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.33913873521882804, 0.1290919628834059, 0.2513384901956327, 0.008072467297792511, 0.0060003533701350874, 32.30686531207067, 22.019219361515525, 5.048271624735861, 0.032323015553043, 2.4785568298725686, 0.0038305787742474767, 2.060970829770957, 0.050138414760259875, 0.0017276320800519854, 0.1644693312720788, 0.825046778674359, 0.0022137006885289464, 4.89900108302614, 11.519930933045995, 15.523572079420145, 0.009377119131974405, 5.112920315946545, 1.9915994320040697, 0.44062851321857943, 1.7940328759054833, 0.05425268730632228, 9.499197835948246, 2.9010949713848224, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 80.42406128262438, 9.686229201178348, 0.7783080689157577, 0.5588886625848235, 0.061470625763865576, 0.6863860251720549, 0.32262537328017105, 0.3355021751343973, 0.20594174058692818, 0.02995108362421354, 0.26128348166608006, 0.3291191965857, 0.04933713516975066, 0.5555410356001004, 0.19690758861767443, 0.1651866082526519, 0.9532606506973565, 0

In [18]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索800-900.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索800-900.xlsx


## 900-1000

In [19]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(900,1000),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.3
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 100.0
alpha = 0.98
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.42201453918841025, 0.6124347339405145, 1.71, 0.011865343511403082, 0.021862608272814398, 54.58011510136781, 25.596979036969483, 5.031126574984256, 0.10857491003819904, 2.9159185157855885, 0.0024823427793562846, 2.5378440270719493, 0.9292876371432489, 0.027442987104498193, 0.2266602982814425, 0.7398360828126557, 4.459264782453896e-06, 0.0006406583039255378, 5.954685450321112, 20.26642802627326, 0.0021061111772074834, 5, 1.0693733049007703, 0.34738223596904394, 1.00164160427122, 0.9782140025887003, 7.791706425409243, 2.105085804161134, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 40.3510445205206, 9.72233504672551, 0.13367887104421897, 0.42723033967509333, 0.7258925206207696, 0.5000743574862794, 0.2531589249984946, 1, 0.3479565615504127, 0.012898333024937824, 0.9857534248131928, 0.8899418490310442, 0.12248715830746243, 0.012966246397922885, 0.5021114889140689, 0.7284206467559609, 0.59298124709964, 0.6261095172335713, 0.778687542826099, 0.941270034

In [20]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索900-1000.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-探索900-1000.xlsx


# 利用

## 500-600

In [7]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(500,600),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [4]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2683889254217945, 0.7762338652992293, 0.003290965041547411, 0.008158541370111566, 0.013768719039603278, 12.188834689997254, 4.738466824373191, 5.08, 0.14897695508981587, 0.19385404777354062, 0.000415320384354857, 2.0562941157676176, 0.07805952821714449, 0.026161977651744703, 0.2507903567093611, 0.5066665586288163, 0.006804534864217869, 1.171410188407694, 13.252917692774034, 5.416415703030737, 0.019707392745948234, 7.487667606192987, 1.9336821296362938, 0.7254151486214366, 1.4213639461023608, 0.9698711733029901, 8.680310094354258, 2.258538625943422, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 66.63938111250822, 4.720017254992124, 0.8747722305523536, 0.9183951747264861, 0.5575942803681657, 0.7987868914873875, 0.4538858974087048, 0.35803487292022995, 0.6306967043037719, 0.4201166314426673, 0.13954894050395336, 0.7403175283673651, 0.945489884739853, 0.7320965187037859, 0.57452530105571, 0.42384993679038163, 0.7647385402136314, 0.6331854970938015, 0

In [5]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-600.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-600.xlsx


In [8]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.3970778978118615, 0.8523872777723845, 0.005417787630584363, 0.018533241905343306, 0.020591966536051463, 32.11546247757758, 13.168609697624508, 0.7249592437868947, 2.2827171712252445, 1.3593160594137526, 0, 1.906117953342182, 1.4731011458900005, 0.28, 0.12734520564634172, 0.35927829252011234, 0.0024317442179159856, 1.844241221241666, 7.702270660712223, 9.09557041358619, 0.012025609925706966, 6.995687909401155, 1, 0.8448249553234379, 1.8676679286276097, 0.9547033701849206, 7.430054447044791, 2.5257490297467577, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 45.73272789426748, 8.025550275419663, 0.7734456264265135, 0.8396913104947724, 0.18837976891344294, 0, 0.3895343916112938, 0.3885393443930391, 0.2141794439030894, 0.2134571927184965, 0.911598885687319, 0.7081333419199886, 0.29910327255740576, 0.7426682536381647, 0.7603864360090893, 0.6080351422582287, 0.47220621921220124, 0.4526853413906803, 0.4089565226502917, 0.28617061904318214, 0.0925543165086

In [9]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6001.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6001.xlsx


In [10]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.13311564407063883, 0.13560837009959495, 0.0022815300638450667, 0.009521563253063704, 0.00678237475846359, 13.132545338563586, 4.606152750193479, 5.068950878423419, 0.7106185592759875, 1.7733977866676238, 0, 0.43489187226658477, 2.1630331738532496, 0.20423717751384562, 0.0016602710296009853, 0.617853219750753, 0.006644506980485701, 2.067584005041306, 0.7240572562358862, 28.88, 0.02212718869124052, 5.921883258862186, 1.2192663447621925, 0.757329670946458, 1.6136708781744773, 0.035415601928779704, 7.852166927869778, 1, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 21.091354923893356, 6.416107877497995, 0.8609652519667839, 0.5417825528736373, 0.7612181190582638, 0.503850982574269, 0.18692780378648677, 0.0346615777374993, 0.5987105949231106, 0.27205946186808166, 0.9237964118251472, 0.9042908956495631, 0.45222438855609715, 0.7971975168168575, 0.22195015768219326, 0.8736643296507304, 0.9787592764883624, 0.7904249974304022, 0.9963334854396487, 0.76865807

In [11]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6002.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6002.xlsx


In [12]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.30649462891400536, 0.9744800091738084, 0.023137588880219617, 0.01606039792536669, 0.021476280366769917, 63.564521894362855, 6.868827064519471, 0.8483804736174366, 1.9090933061860478, 2.912805273162034, 0.0032660755380037832, 2.715884939695973, 0.09563531402402448, 0.17363436782727953, 0.05804454180488613, 0.40781185505721496, 0.0027758618589972285, 0.6443117794633544, 13.005085534921776, 2.0094272764485335, 0.0068920100138179325, 6.625283919310197, 1.900340424982763, 0.9017082491672208, 1.3809378624794943, 0.34540109658823903, 7.756513564724935, 2.262038434933124, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 58.45691703325803, 0.8502834309282223, 0.6257843218209248, 0.3149749140265585, 0.9913390147072404, 0.29052651550823033, 0.5367306552832581, 0.8191659033961709, 0.014797705807265023, 0.034309788497510985, 0.23608768355788412, 0.9195096623809307, 0.3582479033355591, 0.7330679254183657, 0.6204823834854484, 0.14837039583027484, 0.388652178770667

In [13]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6003.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6003.xlsx


In [14]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.42147247210611627, 1.069811059310389, 0.021465664843713546, 0.011197129847171042, 0.023609149432935336, 64.4832578592655, 4.273924643829667, 2.1197298272820753, 2.123346153459235, 1.5286208797267269, 1.9920620102816235e-05, 0.04704835265686768, 2.039245784437647, 0.23702151996963908, 0.12481652975353016, 0.5585121782405328, 0.003495970778692677, 4.399123809783167, 6.278670576181421, 27.21916980081458, 0.002582098186280387, 8.883806196267795, 1.7081889311461702, 0.41478575875379575, 1.2731897743819198, 0.15416474972200395, 6.553728326163897, 2.648435226258352, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 64.00434991251807, 0.17223801176930273, 0.1667565029703883, 0.3696657742316934, 0.5104421520437248, 0.7188294591960113, 0.5936179824150603, 0.5099539417729674, 0.6968993136322182, 0.42620397712198305, 0.6067880442195512, 0.4210599581031654, 0.4222055798790127, 0.8614738252944347, 0.27286862815278773, 0.5373866107168633, 0.6161640455527889, 0.1037

In [15]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6004.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6004.xlsx


In [16]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.10289910259676914, 0.8206983070257013, 0.01138882502028414, 0.008094762245610794, 0.024157915083751768, 64.74751321458433, 14.39326101490744, 0.7294726528348093, 1.1542932495572074, 0.42594070611855767, 0.0033726845379774936, 0.6535968345190519, 0.008084271235785999, 0.026148763894624453, 0.11564952780041476, 0.14633821599748773, 0.007052981571060813, 2.7809580739995288, 5.9111743460775275, 9.975702519963434, 0.0124624423590026, 7.389679304706866, 1.65509866638601, 0.2266069069240147, 1.7098749577557193, 0.32487108015172367, 9.338852333732854, 2.671575364746565, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 44.32095781421341, 2.2196636511505448, 0.7717722588418937, 0.9971190154859993, 0.3865784202179229, 0.0005642653986808271, 0.9958761993949853, 0.10602560842333328, 0.28380450213657316, 0.1986885815883407, 0.0595833435548352, 0.025174822458571607, 1, 0.7385548817468711, 0.29587719820829195, 0.4672969520945879, 0.6562777665414908, 0.7766202705729

In [17]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6005.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用500-6005.xlsx


## 600-700

In [18]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(600,700),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [19]:
n_gen = 50
pop_size = 100
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.1894550162396189, 0.7976042163813912, 1.71, 0.011667880996923116, 0.02044253204015277, 71.69149754869252, 2.078665293356864, 0.6391462697413794, 0.8194077111994963, 1.2606925267969198, 0.004742721697528168, 1.7474149235338872, 0.8876415529263948, 0.026007196726408877, 0.08620018295970314, 0.7461372235567573, 0.009286838103771176, 4.95457302373133, 12.51382915406378, 4.443503760638833, 0.015950009704030092, 6.509411512599672, 1.9872274562373418, 0.24191557260908503, 1.6191051936732153, 0.8680783651151517, 9.869326824026903, 1.8546875596193317, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 40.265342536085456, 7.0262988336931915, 0.11334854440202179, 0.7419402657721623, 0.33969164641343264, 0.5891891023394612, 0.4160522667330813, 0.03737979658071673, 0.7370994030109896, 0.19218148792238302, 0.9180636074419958, 0.731383779512224, 0.06216767325039551, 0.12181758095252737, 0.28479244606048415, 0.08978535266035204, 0.716217038611362, 0.9923567823842527,

In [20]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-700.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-700.xlsx


In [21]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.280950905049926, 0.13954095925736393, 0.12029529553663951, 0.0136736991750335, 0.015969053761479095, 68.6179994725987, 25.539145847941267, 5.065298398362955, 1.0061303817947538, 2.1695747528130966, 0.005205349881223254, 3.0265945982408162, 1.5080011693281414, 0.28, 0.12803328522186974, 0.7563518923974543, 0.008729497048528413, 0.36180406358578254, 12.62729421373088, 21.437340152211956, 0.007693198690061049, 8.485022459445, 1.4443977484408703, 0.2579433244044554, 1.3999914379313703, 0.06469261813976329, 6.1352347785668675, 1.957890844003625, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 80.30008899086653, 9.760083660099651, 0.1142172593058841, 0.20105418747947168, 0.9865852090123822, 0.14709797714835093, 0.21461191813198233, 0.9934230822011562, 0.8184052517553069, 0.9236256261802993, 0.9861853999660541, 0.5019141236639107, 0.4126011918204203, 0.4380794112749467, 0.42961394682258963, 0.5788607089054478, 0.22563622620281099, 0.9440698660528384, 0.57

In [22]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7001.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7001.xlsx


In [23]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.16218301056650347, 0.12714645134252006, 0.9072879524570185, 0.008866027897878511, 0.014971385944247819, 65.87493337733218, 23.619374554953318, 5.069164180621732, 0.7182479676691225, 0.9794055038751184, 0.0033241271222258294, 2.886629689003247, 0.44667771605869777, 0.28, 0.10839977209442025, 0.5553076063339253, 0.009976299810745668, 3.481863007465924, 10.752200580269372, 20.643663284846856, 0.009881376856492255, 5.000000000018774, 1.4821227098585792, 0.3462330735039254, 1.0087934357511077, 0.9359242736702252, 9.58731807327417, 2.01279720014286, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 46.062865999501895, 7.110974344624125, 0.5671871752833275, 0.2203950599781991, 0.47324749775034225, 0.6449461149135899, 0.7401878695319786, 0.8589913326945477, 0.6283411906724368, 0.2852995044967246, 0.724507036999498, 0.11092342682892362, 0.5305435025508867, 0.8648406437611387, 0.07683404876621815, 0.2300334769766419, 0.7791283740876972, 0.9083764194764639, 0.0

In [24]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7002.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7002.xlsx


In [25]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.30798379055302466, 0.6117832680248448, 1.71, 0.009008239073859585, 0.0029115963419347494, 75.11296614264816, 20.192342753805228, 0.5022940406522982, 1.7226837462842655, 0.8963190508504209, 0.0013611320640359037, 0, 2.1882130373951165, 0.02680751474020443, 0.10771726517603442, 0.6643780268402957, 0.003545548038285988, 3.7248769652396154, 11.857891271488784, 21.850336636706658, 0.0005547889983187485, 7.077588051629722, 1.5424620472690622, 0.7917257708672355, 1.6798991012394342, 0.603257629192283, 9.333390558211235, 2.5186362873691746, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 99.56554416228887, 3.925074740523457, 0.578046588329775, 0.4610124279791084, 0.8382780581524875, 0.6468227585463947, 0.5327053530044625, 0.28393234634491876, 0.3967876052074889, 0.9849271158331865, 0.22448258061544793, 0.3958277837288705, 0.9479881490736262, 0.19853445343990492, 0.14582496412572024, 0.11535823359170647, 0.8375112927429869, 0.48698074251115253, 0.7245535947

In [26]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7003.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7003.xlsx


In [27]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.05096856056255428, 0.8341544821110954, 1.71, 0.004109589046206509, 0.01294502100426375, 55.846663366215054, 24.450876528029912, 0.06844884800686826, 0.40956349358611466, 1.7069353782879357, 0.0013283031620950163, 1.265713610799916, 2.1142283227092515, 0.02706014075021805, 0.16577137280026652, 0.14710014225349669, 0.005054641145979782, 3.428884129656973, 5.34619480758317, 10.06742887295338, 0.024924321109247104, 5.762335321659492, 1.3741159923414414, 0.523149375008795, 1.2105334353548718, 0.8351541094737275, 6.0109911417720046, 2.1783369032837134, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 7.867846190791706, 5.980290057452865, 0.9985392835642976, 0.24429587997446628, 0.7927205082509297, 0.7849188105159006, 0.00040079595833447147, 0.667699887715052, 0.42967161331141057, 0.6966786800909973, 0.20470127229922272, 0.6086278430270653, 0.7185534355546086, 0.5468017228076348, 0.42154995043887883, 0.9479261509878136, 0.4460313023215672, 0.47190940740691

In [28]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7004.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7004.xlsx


In [29]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.11996148941912796, 0.1354563922126337, 0.003009232043651929, 0.010894319024786093, 0.006244052972433576, 14.981400560188204, 14.047838933365103, 5.069650038510533, 0.006037026924981908, 2.0099479070099786, 0.002760059793329199, 2.534612550528977, 1.7405099920299447, 0.27999999999896774, 0.246388470364066, 0.770030402566745, 0.005510261793134681, 0.6908922355807933, 5.042974567341751, 5.927257215178837, 0.008171700477316503, 6.417646282241047, 1.0494212541792047, 0.9857706406250737, 1.333162065597152, 0.7357952892885745, 7.144831840593437, 1.0000000003692504, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 29.993210808241628, 9.994863990246754, 0.5174920160640762, 0.7764223567176829, 0.40115526809831464, 0.4733975779680815, 0.6317245892021504, 0.7056455398355203, 0.7589827785741802, 0.00790054851021812, 0.421038868428038, 0.06760220648222463, 0.38250238889397575, 0.7101306922748627, 0.2869017371708902, 0.9567851046246537, 0.7941564279808017, 0.73675

In [30]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7005.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用600-7005.xlsx


## 800-900

In [31]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(800,900),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [32]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.3726208209291327, 0.6586558500412808, 0.0020791944919949495, 0.003937001214168459, 0.02183166125372413, 31.293221393591374, 19.822506487871074, 2.3370441583362127, 0.8194678293351406, 2.918779832931984, 0.0033790732706160125, 0.3944123197274171, 1.3600986425551367, 0.060496493759211435, 0.0619159410940059, 0.7271651867068172, 0.006674495743195769, 2.5740722458572702, 5.666166487136634, 7.194120614660165, 0.023593019940189242, 5.842749910373244, 1.4517555664968305, 0.08910451320452448, 1.5269612665139713, 0.349652857096077, 5.581692749843365, 1.207371799952787, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 24.6724438157431, 3.717045946318328, 0.8815283100454585, 0.09301435033748093, 0.3739438049902823, 0.7154577472744854, 0.7220508353418635, 0.6794306982296985, 0.7493683803766106, 0.7655473230116566, 0.12897796943881767, 0.3826831887116058, 0.9281274084921486, 0.905143160636055, 0.37147015807356437, 0.5958576158905134, 0.771027126025428, 0.3257928

In [33]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-900.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-900.xlsx


In [34]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.41594204271164514, 0.13303197424613847, 0.25043047527489515, 0.0052832082831883054, 0.010242483854260239, 31.048911057892017, 15.382566764725338, 5.068929014012642, 0.3096744355599789, 0.82165823609308, 0.004290360542373972, 3.0520929203452463, 0.4453929909875678, 0.0007039442204777432, 0.07818182740521211, 0.30973235963625445, 0.008808032067958013, 2.3797987734129844, 5.514434057983551, 10.328898715524574, 0.00143852518129963, 7.002250931809726, 1.9316317959477372, 0.5050567708747692, 1.8023504751590738, 0.3321400556362395, 9.847630254447065, 1.939875853360822, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 6.697097963840357e-06, 9.167681349030318, 0.011242022044653834, 0.3437528267579757, 0.44508342612687596, 0.6122446762522212, 0.46231207794228274, 0.1646226470708455, 0.7351500101888558, 0.48262020057244803, 0.15809521734808019, 0.46541620509277076, 0.08983439978098236, 0.3517510159494015, 0.9850294863765511, 0.3706748627016174, 0.9356459072650

In [35]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9001.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9001.xlsx


In [36]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.22422760359393637, 0.6198959045099672, 0.24705919215227393, 0.010686896482423559, 0.0027762268945208334, 4.77694481483986, 8.279783474017654, 2.352769732521952, 1.8689811112273869, 0.769442668441452, 0.0010745649473895976, 2.7567795613722708, 1.0928321061715194, 0.28, 0.0227314204171279, 0.48490989558602954, 0.002836390485500135, 2.8366038277937484, 6.683979819934384, 27.66740874511893, 0.020947198141946753, 5.0791221906748625, 1.2614296243369716, 0.7686931461118917, 1.3763467214524143, 0.8262628574940288, 8.655942987037296, 2.216247768958564, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 62.461803667903, 8.530757446733082, 0.7831970225773613, 0.8554184755593836, 0.867330989546216, 0.9981890744783457, 0.17469165267637915, 0.2715664105068414, 0.18598035423648765, 0.6367495314606036, 0.7611996257010161, 0.15986368683430158, 0.4960895342357401, 1.5334944133878076e-20, 0.10553731236049937, 0.808028289397594, 0, 4.329866399551038e-22, 0.49834221304147

In [37]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9002.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9002.xlsx


In [38]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.1305294846172903, 0.616423007426409, 1.7099999999727526, 0.014707176803768459, 0.014187006522680871, 69.85448481571491, 1.9721108851022717, 5.030144033024861, 2.228944432364276, 0.6941576552150682, 9.605634341709887e-05, 2.993266486588358, 2.3517375595247274, 0.025483998818169908, 0.054258405224043856, 0.4709376640994927, 0.007788035020325573, 1.3904895033471214, 8.714316312589045, 3.7994745610698426, 0.0156614953508809, 6.133181332900303, 1.2986168509143212, 0.052814866334372004, 1.2736008562013976, 0.9224449168403571, 5.826878315355721, 2.647245383582888, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 61.522831223438374, 4.400543739778413, 0.041212248719264204, 0.195478837228347, 0.5717148367340146, 0.8778469117201485, 0.6052995860688111, 0.47940500894988936, 0.17084989620777588, 0.11024098052284685, 0.999841210434559, 0.0037588741339105304, 0.2540512480152217, 0.5869390013647738, 0.6978463641393038, 0.9807011845797876, 0.7336239217648783, 0.879

In [39]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9003.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9003.xlsx


In [40]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.27644209354718396, 0.6124942622948154, 1.7099999953562932, 0.01401514045502071, 0.017755805665218824, 54.0236990789955, 2.2668117934400467, 5.0468025742693605, 0.4346137755282525, 2.4990463057578527, 0.0057538179550758175, 2.4195801189630055, 1.9497684088363503, 0.026149611186658945, 0.10718195359851651, 0.37716680407426906, 0.0009607023869516343, 1.0011641762398558, 9.713900390993667, 18.19345786666657, 0.026755763843981102, 8.567513434771653, 1.8776723876333068, 0.15993719774886575, 1.7072502060446386, 0.2537906945732374, 5.8529289554850985, 2.8737203801136775, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 12.709814421589346, 5.092558922617018, 0.2685979169879774, 0.3765788469231747, 0.3401130079813946, 0.009587157078329394, 1.9294554438948793e-07, 0.9572359429468518, 0.6602781933037655, 0.01321341093749451, 0.9580323224153041, 0.8386275922744952, 0.00366105826433303, 0.13270461293169689, 0.5632657056946143, 0.40228043969977706, 0.5383267998074

In [41]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9004.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9004.xlsx


In [42]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.1397161823641937, 0.11958518567394698, 0.10402066891015953, 0.006985364973186655, 0.025999999999988352, 2.9944562303477906, 23.252154717430276, 5.039470538314079, 2.456762951100536, 2.6801319075895873, 0.001861062273409656, 2.283664298737176, 1.2099955966566072, 0.0007031525342443056, 0.0019018505821271335, 0.6882842043688612, 0.004695585581053721, 2.794566439268653, 6.411671849091572, 0.19475993924672422, 0.01825001072325532, 8.92163783767659, 1.767910157083657, 0.13873722383068204, 1.3581187983034588, 0.6157190346060119, 6.847502945906043, 2.9501052629912112, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 35.665502420020815, 5.536135914064824, 0.5245426927631253, 0.5418214761563191, 0.4613619837104744, 0.205388066531793, 0.041695690885635114, 0.399516950811697, 0.13823642837096012, 0.34107732857876555, 0.4400990977397008, 0.4753728842736003, 0.9915730875051997, 0.47765242004151437, 0.5354818063947885, 0.9781098976123951, 0.5264644222760785, 0.17

In [43]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9005.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用800-9005.xlsx


## 700-800

In [4]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(700,800),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [5]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.08155508109212127, 0.6167711688114229, 1.71, 0.008051810172524702, 0.014188383490186404, 29.32721480144879, 1.7994510963680863, 5.050461476761786, 1.2304475667735146, 2.8211206304209444, 0.0037136040600778066, 2.5794471097102387, 1.0467702103164755, 0.02567208253206118, 0.1794620864628604, 0.2901233298990413, 0.009127628708485443, 2.8920482054823204, 9.727465152325381, 23.403008359715972, 0.00010758410691680758, 5.489123951844996, 1.9225724018376407, 0.38053252975249163, 1.6000160110576982, 0.20868969863762746, 6.271708865664853, 1.5240131305464797, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 11.550931327545184, 5.654978061956677, 0.4244593173017001, 0.3271496213329707, 0.8160623951252639, 0.38423890096437086, 0.4223545125642004, 0.04443127852896104, 0.20833490303224173, 0.7891722898916851, 0.21747776152555368, 0.005675284579114753, 0.3284748811114387, 0.03961535655589357, 0.2675152256484473, 0.7507739908560843, 0.43916414214979194, 0.675989472

In [6]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-800.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-800.xlsx


In [7]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2542005664689446, 0.8912555457344035, 1.71, 0.022629889647820783, 0.016999801971822666, 68.29428865163521, 22.67768109929914, 5.039181022209648, 2.067114563009024, 2.919999999999999, 0.005760163233581952, 2.218941388931996, 0.7496051521932121, 0.02624188990438297, 0.17792322574722413, 0.8718450329052775, 0.0026738800919261295, 0.24110265149112012, 8.227726222408299, 15.775124762464431, 0.022773624993486296, 6.369868386466344, 1.309584392176917, 0.44215937475293987, 1.3589290809576726, 0.3953981570936083, 6.536268397412742, 2.0740247989288028, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 26.200133880608707, 0.5407850121927338, 0.7925771276614441, 0.17883607323854442, 0.07441433269316713, 0.17563056237388733, 0.8724062873660255, 0.5550401212721333, 0.5932136886887772, 0.5183586206679199, 0.16425397164070754, 0.8035392263651977, 0.037470600612349705, 0.7072432866585895, 0.2707741104075696, 0.05323833222908969, 0.6508438587016278, 0.0925977462172755

In [8]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8001.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8001.xlsx


In [9]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.43, 0.6170705451344753, 1.71, 0.0031155748227929907, 0.0010115700999086654, 62.084872453391384, 10.34024659862079, 5.055794259375763, 2.463768496801788, 1.8196844773522727, 0.0033054873637992713, 2.1991022139241974, 2.0377959510909407, 0.02502820394253017, 0.041413175741723825, 0.592179037601689, 0.008235091869932133, 2.5385990645647154, 14.031222034975363, 4.040616073891734, 0.008135627676216598, 6.8666295849634995, 1.0302364393444583, 0.14363842183504394, 1.770542466736193, 0.6445888398968456, 6.028540043980498, 2.2354860172325774, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 17.12518641703668, 8.489196954871396, 0.02822752286615524, 0.13311573061341708, 0.4530155698394655, 0.16890543291711707, 0.08616074242253607, 0, 0.6809233773811276, 0.3107208283900642, 0.5887614777262128, 0.6609916605006434, 0.5630620012418506, 0.046086979661721615, 0.07928043916406242, 0.9672096797640317, 0.8228155259150225, 0.2766376363108898, 0.007063747755918095, 0.89

In [10]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8002.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8002.xlsx


In [11]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.25499899535623755, 0.6160780573607598, 1.7099999985723997, 0.012974122563993813, 0.0020275819166191487, 72.39850384086256, 0.04000000465002825, 5.0311387851341, 1.0939985937785262, 2.9199968203975923, 0.004223965326642075, 3.0585114628735477, 2.3010060000895542, 0.026228081183070136, 0.08519209653070452, 0.640336469013435, 0.005368519378492168, 0.8926253671336849, 4.057007505105877, 22.47332850562504, 0.0030542288596077547, 5.886472950370909, 1.4782391903385805, 0.11086842345506509, 1.2120601790095844, 0.47923655602664267, 5.8107115483197145, 1.026076435802131, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 88.98814199927729, 6.43078006891956, 0.7369372367662409, 0.18668560208767115, 0.16894887665675762, 0.9990291144089104, 0.32961775303278973, 0.7812047665302285, 0.32179975742907996, 0.27785700032271277, 0.5921922075071002, 0, 0.9579312356480997, 0.7326032477222343, 0.12253741307926747, 0.861893790475623, 0.0484024491619228, 0.9923159264223653, 0

In [12]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8003.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8003.xlsx


In [13]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.32082939885472567, 0.6199127525322563, 0.7800955560762173, 0.004966088480785443, 0.016687292528272467, 44.69182792262766, 21.253661689616216, 5.039023772847459, 1.9757833369752107, 2.3020172009137037, 0.0011722049220836575, 0.9391430927195513, 1.2783464177572645, 0.027467356597451396, 0.25760758243037446, 0.09318838507342332, 0.0008849292901410373, 4.701458455968049, 3.0600304153672657, 15.33259763148029, 0.006463956802173433, 8.771215652669802, 1.6458192964368807, 0.3698780811579442, 1.0980599127337758, 1, 10, 1.6811441860898275, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 77.40688392770484, 7.553153179806183, 0.7287837704908674, 0.1884221722543542, 0.09269270072341514, 0.9999996426014153, 0.600448533581023, 0.6905135379184271, 0.7278951399173104, 0.4653221150798719, 0.9992724253525771, 0.32111907457385513, 0.740009347256938, 0.7250391845582516, 0.7026258011006205, 0.03612036421037748, 0.06785363857721938, 0.7982892809087389, 0.540139298015987

In [14]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8004.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8004.xlsx


In [15]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.4123858539539616, 0.03270325627809216, 0.19324007848653543, 0.002000001197258576, 0.01735106472172986, 40.74485761226548, 22.141800199727303, 5.0384951095552495, 1.5847566786606049, 2.341619003834233, 0.005228423093009741, 0.30266534009093965, 0.01909737211538624, 0.2799999992749417, 0.24736291339504451, 0.7231304554930018, 0.004629606812392979, 0.17181814785508873, 2.320863541130348, 24.476783844105285, 0.0023491302523358224, 6.866561176956384, 1.968832105488333, 0.05163591911957347, 1.0919780128830106, 0.19435639531444635, 8.310407991465961, 2.999999997050133, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 66.11315791363627, 2.4903879143440983, 0.41677554670963646, 0.5479300367246887, 0.9818581140547643, 0.8096225014303271, 0.46530631745104956, 0.47893802585095857, 0.774350542438576, 0.9782058124158818, 0.8986744631295632, 0.6841527057695105, 0.22899142110448567, 0.43772303492751624, 0.2304692082213546, 0.4911330007505088, 0.7196295133772628, 0.

In [16]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8005.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用700-8005.xlsx


## 900-1000

In [17]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(900,1000),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [18]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.3527024269488201, 0.9579129082262655, 1.7099999999999789, 0.02218241701370511, 0.015556818714225717, 72.87934631200119, 16.14024332778356, 5.048612364439006, 2.3402552006092177, 1.436299311086833e-10, 0.0009399916931385516, 2.8383220847700645, 0.4215136537655878, 0.025683242491091097, 0.15409160561828575, 0.037543532938733484, 0.0049919377560351565, 4.091044619381118, 13.89808097699197, 26.127776410989977, 0.021949232205684142, 8.566577912800023, 1.4577861797901155, 0.4094815322245506, 1.116785215294877, 0.367777476813575, 9.449851797949327, 1.9678004513600262, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 56.10644854612204, 8.616122614425977, 0.595832500418074, 0.999999992786045, 0.7229942783483935, 0.4168739572008816, 0.6642154828767564, 0.9116560291412784, 0.10139405005328966, 0.3002696713115666, 0.8165901752551153, 0.29030023346866735, 0.4588939934497894, 0.3228264034833007, 0.8494524913661681, 0.6649447576590187, 0.2396106210479825, 0.837044

In [19]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-1000.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-1000.xlsx


In [20]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.23853134972881013, 0.12002572033318314, 0.0892346808913778, 0.00530001186464615, 0.013935566561206307, 35.549021821716984, 6.922465369975359, 5.033519100397247, 1.1333731133384324, 2.6480128905066715, 0.0048188705675029815, 0.5595669681127559, 0.34080242625307433, 0.017989625875568186, 0.06419295290030246, 0.8440453459581071, 0.005596898738790773, 0.08262725343002204, 8.827566443829355, 23.1744593863386, 0.0010637691828356633, 8.736419053737677, 1.000000001997499, 0.3613853059681902, 1.999999998396954, 0.8322583924979073, 7.346574724705987, 2.4712284445679185, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 45.409657639513064, 5.637855195159931, 0.8514263735359823, 0.4144624228106833, 0.38967825146815277, 0.6063312432360761, 0.404943571165839, 0.3562795948634023, 0.764203376426858, 0.4292023345729523, 0.9653040601345615, 0.964269644994376, 0.2679656574766676, 0.5481052912714567, 0.21597239852045344, 0.875154562816002, 0.5354454099497755, 0.99286316

In [21]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10001.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10001.xlsx


In [22]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.85
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.1160313475765346, 1.0177602407257997, 0.4290958024754186, 0.005699763699005509, 0.017459801266162107, 65.10704618584354, 8.435158314651055, 5.027921161093963, 1.4765778218834666, 2.8948137227995465, 0.0016651496513456873, 0.3713352807052116, 1.1734235555727643, 0.026403294849642184, 0.09421300874536491, 0.31477046318287066, 0.003498255495429452, 3.309279528169153e-17, 3.010685654292333, 18.461611966056893, 0.02626490258495657, 5.261600717999268, 1.4098671288364357, 0.02129468036630694, 1.2995643214552977, 0.9792023743861761, 9.323886517020826, 1.556514200674758, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 17.955261808038934, 0.15821486243285587, 0.00622258602228608, 0.2719137980947086, 0.9052584966666405, 0.39725516519268345, 0.7977854131402696, 0.19498267103314257, 0.4905324874116225, 0.2993960029502335, 0.625293542960873, 0.49155103038984527, 0.6759785861656527, 0.7023284682025343, 0.9282664793880541, 0.5825918797525623, 0.6281924600338704, 0

In [23]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10002.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10002.xlsx


In [24]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.7, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2626534142180477, 0.6153539038509476, 0.0022254249257269154, 0.012655106980438598, 0.009152080474641496, 57.12274867948513, 6.541059314778055, 2.3370360921282436, 1.991808198058123, 1.0442479012581551, 0.0033406621573768497, 1.264856325069224, 1.7421039581857662, 0.06639236413958671, 0.13194127080446688, 0.29503136547142855, 0.008436574436509843, 3.917416141971177, 12.734874979547339, 7.410987937838225, 0.0016422509950834337, 8.606717116053437, 1.0515812859946907, 0.9666486557662449, 1.3511855103454182, 0.8281325808883232, 9.671558934895607, 2.3606402664859143, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 72.24446821788098, 9.58232886365412, 0.47953770851353716, 2.9540971794013344e-08, 0.7774506971402279, 0.9999527676761516, 0.992294432680219, 0.49755715087450136, 0.30592377696371187, 0.024865114862953463, 0.17154107393085857, 0.9941199299750155, 0.9291174288047734, 0.13528597639177706, 0.6545835518265347, 0.6693120971320835, 0.2907690974412972,

In [25]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10003.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10003.xlsx


In [26]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.6, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.27064690952759674, 0.6146464340167188, 1.71, 0.00607242603150009, 0.025435929206965222, 74.0113068247199, 24.429097804419552, 0.964056405341451, 0.1569641079724972, 1.6968690152569679, 0.001683516117029269, 1.4117731403893161, 0.34776852501713074, 0.025616189837771433, 0.187258905202406, 0, 0.008872031350617473, 0.9223231739059818, 7.817132292719025, 25.972704116811702, 0.019809082327787768, 6.590428926624857, 1.1230571325936136, 0.23964138595786602, 1.3966216318804714, 0.3851454609702241, 9.232237600759953, 1.5376819629736678, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 62.229013213975136, 4.367411453108469, 0.6288464217193652, 0.9599188226510625, 0.10428006008158766, 0.8711160673141586, 0.282115990459149, 0.010643272802986265, 0.10794720884151499, 0.4610817358402981, 0.7900127978088343, 0.11453909699576363, 0.7551353053855179, 0.110592806000399, 0.8031376197586232, 0.780507255946811, 0.21349050390039806, 0.5857160235561647, 0.4975382685381417

In [27]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10004.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10004.xlsx


In [28]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 70.0
alpha = 0.90
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2923869472727104, 0.7702294369601111, 1.7099999965738912, 0.020655194774735215, 0.0071460911638005905, 45.13982823344087, 6.500460542084093, 5.055303504001653, 1.5697254498268263, 0.21250909192630926, 0.0042802141967159466, 2.2882943409462215, 0.9714684371422925, 0.0007058689193383921, 0.027804115127423645, 0.5619845753037473, 0.00906645565187051, 4.334176610811976, 9.818636685030773, 10.185030725954611, 0.008952508531953916, 5.7431062009552205, 1.6218137005607032, 0.3216165572378528, 1.009700122925869, 0.04904400379417349, 6.484714433170736, 1.0002675986210214, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 5.246926751815819, 7.824394765988009, 0.5610346727189544, 0.41613697407504274, 0.9393955180127809, 0.05018264255552825, 0.9152447325465302, 0.15615564784026967, 0.4655810403624255, 0.5573851234224675, 0.39252453516913405, 0.9127909743419826, 6.005546136250669e-09, 0.9999999995309871, 0.9934970534829771, 0.15031161736588194, 0.9999999790382124,

In [29]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10005.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-利用900-10005.xlsx


# 平衡

In [15]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.02,1.2),(0.002,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,5.08),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[0,0],[0,0],[0,0],[0,0],[1050,1050],[0.42,0.42],[640,640],[1,1],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(900,1000),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalTR.predict(x_matrix)[0]
    val_TS = model_optimalTs.predict(x_matrix)[0]
    return val_TR, val_TS
# SA变异函数
def mutate_SA(individual, min, max, indpb, temperature):
    for i in range(len(individual)):
        if random.random() < indpb:
            perturb = (max[i] - min[i]) * temperature / 100.0
            individual[i] += random.uniform(-perturb, perturb)
            individual[i] = min[i] if individual[i] < min[i] else individual[i]
            individual[i] = max[i] if individual[i] > max[i] else individual[i]
    return individual,

min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate_SA, min=min_values, max=max_values, indpb=0.2, temperature=1.0) # 修正为mutate_SA
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [16]:
n_gen = 500
pop_size = 200
CXPB, MUTPB = 0.5, 0.2
def annealing_schedule(start_temp, alpha, n_generations):
    temp = start_temp
    for _ in range(n_generations):
        yield temp
        temp *= alpha
    # 使用 itertools.cycle 循环使用温度序列
    return itertools.cycle([temp])

initial_temperature = 90.0
alpha = 0.95
schedule = annealing_schedule(initial_temperature, alpha, n_gen)
population = toolbox.population(n=pop_size)
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(n_gen):
    temperature = next(schedule)

    offspring = algorithms.varOr(population, toolbox, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB)
    for ind in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(ind, temperature=temperature) # 这里改为原生mutate
            del ind.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(population + offspring, k=pop_size)

# 打印 Pareto 前沿数据
print("Pareto front:")
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

Pareto front:
Composition: [0.2686799038288577, 0.1081819260431328, 0.2460730999703048, 0.005379527689728378, 0.0069456747670238765, 62.712066177155215, 4.5891935144948075, 5.036900841632424, 1.4508828433080676, 1.4118602292810392, 0.0019187657276618342, 0.7584919522076886, 2.1968911691415873, 0.00172728122013204, 0.1324901831273949, 0.06514676755377602, 0.002344911952004217, 1.3761043876012395, 0.9034014343587584, 3.6170228318794906, 0.0005472995678801713, 6.61810102490478, 1.735949321553469, 0.8015188348735431, 1.0664631790096348, 0.6174999913782603, 9.04996843112678, 1.0567534757097814, 0.0, 0.0, 0.0, 0.0, 1050.0, 0.42, 640.0, 1.0, 0.0, 0.0, 70.91664725083984, 2.373992242119716, 0.9971219784608853, 0.9058519527834872, 0.45905031070047303, 0.9999999874297139, 0.6563246966233528, 0.11398509679270173, 0.9798277104454345, 0.08480124194985383, 0.6538465193807261, 0.35485980772946735, 0.6860425828444207, 0.4882118459239556, 0.9955512561960047, 0.1183008709683166, 0.9898426492485068, 0.620

In [17]:
pareto_front_data = []
for ind in tools.sortNondominated(population, len(population), first_front_only=True)[0]:
    pareto_front_data.append(list(ind) + list(ind.fitness.values))

pareto_front_df = pd.DataFrame(pareto_front_data, columns=element_names + ["TR Value", "TS Value"])

# 保存 DataFrame 到 Excel 文件
excel_file_path = r'D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-平衡900-1000.xlsx'
pareto_front_df.to_excel(excel_file_path, index=False)

print(f"Pareto front saved to: {excel_file_path}")

Pareto front saved to: D:\creep rupture\TL\外推后的多目标寻优\NSSA\A-平衡900-1000.xlsx
